#### **Senior International Match Feature Table**

- Goal: build the base match-level modeling table used by the downstream ranking, squad, EDA, and model notebooks. Each row is one senior national-team match, represented from the home team perspective with matching away-team columns.

- The features are restricted to information available before kickoff: draw context, stadium and schedule context, published squad information, and historical World Cup performance from earlier tournaments only.


In [1]:
from __future__ import annotations

import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd


def repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "README.md").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not find repo root with README.md and data/.")


REPO_ROOT = repo_root()
RAW_DIR = REPO_ROOT / "data" / "raw"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
DB_PATH = RAW_DIR / "worldcup" / "data-sqlite" / "worldcup.db"

assert DB_PATH.is_file(), f"SQLite DB not found: {DB_PATH.relative_to(REPO_ROOT)}"

con = sqlite3.connect(DB_PATH)


def read_sql(q: str) -> pd.DataFrame:
    return pd.read_sql_query(q, con)


read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")

,name
0,award_winners
1,awards
2,bookings
3,confederations
4,goals
5,group_standings
6,groups
7,host_countries
8,manager_appearances
9,manager_appointments


**Load source tables**

Read the match, team, player, coach, and stadium tables we need for the steps below.

In [2]:
matches = read_sql("SELECT * FROM matches")
tournaments = read_sql("SELECT * FROM tournaments")
teams = read_sql("SELECT * FROM teams")
confederations = read_sql("SELECT * FROM confederations")
stadiums = read_sql("SELECT * FROM stadiums")
team_appearances = read_sql("SELECT * FROM team_appearances")
squads = read_sql("SELECT * FROM squads")
players = read_sql("SELECT * FROM players")
manager_appointments = read_sql("SELECT * FROM manager_appointments")
managers = read_sql("SELECT * FROM managers")
manager_appearances = read_sql("SELECT * FROM manager_appearances")
referee_appearances = read_sql("SELECT * FROM referee_appearances")

tournaments["start_dt"] = pd.to_datetime(
    tournaments["start_date"], unit="D", origin="unix", errors="coerce"
)
matches["match_dt"] = pd.to_datetime(
    matches["match_date"], unit="D", origin="unix", errors="coerce"
)

tourn_year = tournaments[["tournament_id", "year", "host_country", "start_dt"]].copy()
matches = matches.merge(tourn_year, on="tournament_id", how="left", suffixes=("", "_t"))

matches.shape, team_appearances.shape

/var/folders/rw/9x0mnb0n50zdfcqqs7nwzg8m0000gn/T/ipykernel_98070/2987988674.py:17: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  matches["match_dt"] = pd.to_datetime(


((1248, 32), (2496, 17))

**Sanity checks**

A short sanity pass: how big each table is, which years are covered, whether dates and keys look usable, and a rough label distribution (`home_team_win`).

In [3]:
from IPython.display import display

summary = pd.DataFrame(
    {
        "rows": {
            "matches": len(matches),
            "tournaments": len(tournaments),
            "team_appearances": len(team_appearances),
            "squads": len(squads),
            "players": len(players),
            "teams": len(teams),
            "manager_appointments": len(manager_appointments),
            "manager_appearances": len(manager_appearances),
            "stadiums": len(stadiums),
        }
    }
).sort_values("rows", ascending=False)
display(summary)

y0, y1 = int(tournaments["year"].min()), int(tournaments["year"].max())
print(
    f"Tournament years: {y0} – {y1}  ({tournaments['tournament_id'].nunique()} tournaments)\n"
)

ratio = len(team_appearances) / max(len(matches), 1)
print(
    f"team_appearances / matches = {ratio:.3f}  (expect ~2.0: home + away row per game)\n"
)

mpt = matches.groupby("tournament_id").size()
print("Matches per tournament:")
display(mpt.describe().to_frame("matches").T)

key_m = [
    "match_date",
    "home_team_id",
    "away_team_id",
    "stadium_id",
    "home_team_win",
]
print("Missing share in `matches` (key columns):")
display(matches[key_m].isna().mean().round(4).to_frame("missing_share"))

print(
    f"Share of matches with a parsed `match_dt`: {matches['match_dt'].notna().mean():.4f}\n"
)

print("home_team_win (1 = home won; used later as y_win):")
display(matches["home_team_win"].value_counts(dropna=False).to_frame("n"))

sq_sz = squads.groupby(["tournament_id", "team_id"]).size()
print("Squad size (number of players) per team per tournament:")
display(sq_sz.describe().round(2).to_frame("players"))

ma_n = manager_appointments.groupby(["tournament_id", "team_id"]).size()
print(
    f"Manager rows per team-tournament: min={ma_n.min()}, median={ma_n.median():.0f}, max={ma_n.max()}"
)

,rows
squads,13843
players,10401
manager_appearances,2538
team_appearances,2496
matches,1248
manager_appointments,637
stadiums,240
teams,88
tournaments,30


Tournament years: 1930 – 2022  (30 tournaments)

team_appearances / matches = 2.000  (expect ~2.0: home + away row per game)

Matches per tournament:


,count,mean,std,min,25%,50%,75%,max
matches,30.0,41.6,16.304431,17.0,32.0,36.5,52.0,64.0


Missing share in `matches` (key columns):


,missing_share
match_date,0.0
home_team_id,0.0
away_team_id,0.0
stadium_id,0.0
home_team_win,0.0


Share of matches with a parsed `match_dt`: 1.0000

home_team_win (1 = home won; used later as y_win):


,n
home_team_win,
1,703
0,545


Squad size (number of players) per team per tournament:


,players
count,625.00
mean,22.15
std,1.54
min,15.00
25%,22.00
50%,22.00
75%,23.00
max,26.00


Manager rows per team-tournament: min=1, median=1, max=2


**Match-level base table**

We keep one row per game. The outcome column for modeling is `y_win`: `1` if the home team won, `0` if the match was a draw or the away team won.

In [4]:
base = matches[
    [
        "match_id",
        "tournament_id",
        "year",
        "host_country",
        "home_team_id",
        "away_team_id",
        "match_dt",
        "stage_name",
        "group_name",
        "group_stage",
        "knockout_stage",
        "replayed",
        "replay",
        "stadium_id",
        "home_team_win",
    ]
].copy()

base["y_win"] = base["home_team_win"].astype("Int64").fillna(0).astype(int)

th = teams.rename(columns={c: f"home_{c}" for c in teams.columns if c != "team_id"})
ta = teams.rename(columns={c: f"away_{c}" for c in teams.columns if c != "team_id"})
base = base.merge(th, left_on="home_team_id", right_on="team_id", how="left").drop(
    columns=["team_id"]
)
base = base.merge(ta, left_on="away_team_id", right_on="team_id", how="left").drop(
    columns=["team_id"]
)

base = base.merge(stadiums, on="stadium_id", how="left", suffixes=("", "_stadium"))

base["feat_same_confederation"] = (
    base["home_confederation_id"] == base["away_confederation_id"]
).astype(int)


def _norm_host(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.lower().str.replace(" ", "", regex=False)


base["feat_home_is_host_nation"] = (
    _norm_host(base["home_team_name"]) == _norm_host(base["host_country"])
).astype(int)

base["feat_away_is_host_nation"] = (
    _norm_host(base["away_team_name"]) == _norm_host(base["host_country"])
).astype(int)

base.shape

(1248, 45)

**Prior World Cup team strength**

- Counts and rates use only games from tournaments with `year` less than this row’s tournament year—nothing from the current cup’s results.

- Smoothed win rate: we use \((w + 2) / (n + 4)\) so teams with very few past games do not look like 0% or 100% winners by chance.

In [5]:
ta_hist = team_appearances.merge(
    tourn_year[["tournament_id", "year"]], on="tournament_id", how="left"
)
ta_hist = ta_hist.merge(
    matches[["match_id", "knockout_stage", "group_stage"]],
    on="match_id",
    how="left",
    suffixes=("", "_m"),
)

ALPHA, BETA = 2.0, 2.0


def prior_team_stats(team_id: str, before_year: int) -> dict:
    sub = ta_hist[(ta_hist["team_id"] == team_id) & (ta_hist["year"] < before_year)]
    if sub.empty:
        return {
            "n_tournaments": 0,
            "n_matches": 0,
            "wins": 0,
            "draws": 0,
            "losses": 0,
            "goal_diff_sum": 0,
            "ko_matches": 0,
            "ko_wins": 0,
            "pso_matches": 0,
            "pso_wins": 0,
            "et_matches": 0,
            "tournaments_with_ko_game": 0,
        }
    n_tournaments = sub["tournament_id"].nunique()
    wins = int(sub["win"].sum())
    dr = int(sub["draw"].sum())
    lo = int(sub["lose"].sum())
    n_matches = len(sub)
    gd = float((sub["goals_for"] - sub["goals_against"]).sum())
    ko = sub[sub["knockout_stage"] == 1]
    ko_matches = len(ko)
    ko_wins = int(ko["win"].sum())
    pso = sub[sub["penalty_shootout"] == 1]
    pso_matches = len(pso)
    pso_wins = int((pso["penalties_for"] > pso["penalties_against"]).sum())
    et_matches = int(sub["extra_time"].sum())
    ko_tids = sub.loc[sub["knockout_stage"] == 1, "tournament_id"].unique()
    tournaments_with_ko_game = len(set(ko_tids.tolist()))
    return {
        "n_tournaments": n_tournaments,
        "n_matches": n_matches,
        "wins": wins,
        "draws": dr,
        "losses": lo,
        "goal_diff_sum": gd,
        "ko_matches": ko_matches,
        "ko_wins": ko_wins,
        "pso_matches": pso_matches,
        "pso_wins": pso_wins,
        "et_matches": et_matches,
        "tournaments_with_ko_game": tournaments_with_ko_game,
    }


def shrink_win_rate(w: int, n: int, a: float = ALPHA, b: float = BETA) -> float:
    if n == 0:
        return a / (a + b)
    return (w + a) / (n + a + b)


pairs = base[["home_team_id", "away_team_id", "year"]].drop_duplicates()
team_year_keys = pd.unique(
    np.concatenate([pairs["home_team_id"].values, pairs["away_team_id"].values])
)
years = sorted(base["year"].dropna().unique())

cache: dict[tuple[str, int], dict] = {}
for tid in team_year_keys:
    for y in years:
        cache[(tid, int(y))] = prior_team_stats(tid, int(y))


def attach_hist(df: pd.DataFrame, prefix: str, team_col: str) -> pd.DataFrame:
    rows = []
    for _, r in df.iterrows():
        st = cache[(r[team_col], int(r["year"]))]
        rows.append(st)
    h = pd.DataFrame(rows)
    h = h.add_prefix(f"{prefix}hist_")
    h[f"{prefix}hist_win_rate_shrunk"] = [
        shrink_win_rate(int(w), int(n))
        for w, n in zip(h[f"{prefix}hist_wins"], h[f"{prefix}hist_n_matches"])
    ]
    h[f"{prefix}hist_goal_diff_per_match"] = np.where(
        h[f"{prefix}hist_n_matches"] > 0,
        h[f"{prefix}hist_goal_diff_sum"] / h[f"{prefix}hist_n_matches"],
        np.nan,
    )
    h[f"{prefix}hist_ko_win_rate_shrunk"] = [
        shrink_win_rate(int(kw), int(km))
        for kw, km in zip(h[f"{prefix}hist_ko_wins"], h[f"{prefix}hist_ko_matches"])
    ]
    h[f"{prefix}hist_pso_win_rate_shrunk"] = [
        shrink_win_rate(int(pw), int(pm)) if pm > 0 else np.nan
        for pw, pm in zip(h[f"{prefix}hist_pso_wins"], h[f"{prefix}hist_pso_matches"])
    ]
    h[f"{prefix}hist_et_rate"] = np.where(
        h[f"{prefix}hist_n_matches"] > 0,
        h[f"{prefix}hist_et_matches"] / h[f"{prefix}hist_n_matches"],
        np.nan,
    )
    h[f"{prefix}hist_frac_tournaments_reached_ko"] = np.where(
        h[f"{prefix}hist_n_tournaments"] > 0,
        h[f"{prefix}hist_tournaments_with_ko_game"] / h[f"{prefix}hist_n_tournaments"],
        np.nan,
    )
    return pd.concat([df.reset_index(drop=True), h], axis=1)


base = attach_hist(base, "home_", "home_team_id")
base = attach_hist(base, "away_", "away_team_id")

base["feat_hist_win_rate_diff"] = (
    base["home_hist_win_rate_shrunk"] - base["away_hist_win_rate_shrunk"]
)
base["feat_hist_goal_diff_per_match_diff"] = (
    base["home_hist_goal_diff_per_match"] - base["away_hist_goal_diff_per_match"]
)

**Prior performance by opponent region**

Using again only older World Cups, we summarize win rate (smoothed) when facing teams in the same confederation as today’s opponent. Small samples can be noisy; that is why smoothing matters.

In [6]:
opp_conf = teams[["team_id", "confederation_id"]].rename(
    columns={"team_id": "opponent_id", "confederation_id": "opponent_confederation_id"}
)
ta_oc = ta_hist.merge(opp_conf, on="opponent_id", how="left")


def rate_vs_conf(team_id: str, before_year: int, opp_conf_id: str | float) -> float:
    if pd.isna(opp_conf_id):
        return np.nan
    sub = ta_oc[
        (ta_oc["team_id"] == team_id)
        & (ta_oc["year"] < before_year)
        & (ta_oc["opponent_confederation_id"] == opp_conf_id)
    ]
    if sub.empty:
        return np.nan
    return shrink_win_rate(int(sub["win"].sum()), len(sub))


base["feat_home_hist_win_rate_vs_away_conf"] = [
    rate_vs_conf(h, int(y), c)
    for h, y, c in zip(
        base["home_team_id"], base["year"], base["away_confederation_id"]
    )
]
base["feat_away_hist_win_rate_vs_home_conf"] = [
    rate_vs_conf(a, int(y), c)
    for a, y, c in zip(
        base["away_team_id"], base["year"], base["home_confederation_id"]
    )
]

**Squad makeup and experience**

- From `squads` + `players`: ages (vs tournament start), position counts, share of players who already played in a World Cup before this one. 

- Column `prior_wc_played` counts how many distinct past World Cups that player was in (only `year <` current tournament year). 

- We also measure overlap with the same country’s previous World Cup squad (Jaccard and raw overlap count).

In [7]:
sq = squads.merge(tourn_year[["tournament_id", "year", "start_dt"]], on="tournament_id")
sq = sq.merge(
    players,
    on="player_id",
    how="left",
)
sq["birth_dt"] = pd.to_datetime(sq["birth_date"], errors="coerce")

player_tourney_years = sq[["player_id", "tournament_id", "year"]].drop_duplicates()

prior_rows = []
for (pid, y), g in player_tourney_years.groupby(["player_id", "year"]):
    n_prior = player_tourney_years[
        (player_tourney_years["player_id"] == pid) & (player_tourney_years["year"] < y)
    ]["tournament_id"].nunique()
    prior_rows.append({"player_id": pid, "year": y, "prior_wc_played": n_prior})

prior_wc = pd.DataFrame(prior_rows)
sq = sq.merge(prior_wc, on=["player_id", "year"], how="left")

sq["age_at_tournament"] = (sq["start_dt"] - sq["birth_dt"]).dt.days / 365.25

for col in ["goal_keeper", "defender", "midfielder", "forward"]:
    sq[col] = sq[col].fillna(0).astype(int)


def squad_row(g: pd.DataFrame) -> dict:
    return {
        "n_players": len(g),
        "age_mean": g["age_at_tournament"].mean(),
        "age_median": g["age_at_tournament"].median(),
        "age_std": g["age_at_tournament"].std(),
        "age_min": g["age_at_tournament"].min(),
        "age_max": g["age_at_tournament"].max(),
        "n_gk": int(g["goal_keeper"].sum()),
        "n_df": int(g["defender"].sum()),
        "n_mf": int(g["midfielder"].sum()),
        "n_fw": int(g["forward"].sum()),
        "prior_wc_mean": g["prior_wc_played"].mean(),
        "prior_wc_median": g["prior_wc_played"].median(),
        "share_any_prior_wc": (g["prior_wc_played"] > 0).mean(),
        "share_ge2_prior_wc": (g["prior_wc_played"] >= 2).mean(),
    }


squad_records = []
for (tournament_id, team_id), g in sq.groupby(["tournament_id", "team_id"]):
    row = {"tournament_id": tournament_id, "team_id": team_id, **squad_row(g)}
    squad_records.append(row)
squad_feats = pd.DataFrame(squad_records)

overlap_records = []
for (team_id,), g in sq.groupby(["team_id"]):
    g = g.sort_values("year")
    years_order = g["year"].unique()
    prev_players: set[str] | None = None
    prev_year: int | None = None
    for y in years_order:
        cur = set(g.loc[g["year"] == y, "player_id"].unique())
        if prev_players is not None and prev_year is not None:
            inter = len(cur & prev_players)
            uni = len(cur | prev_players)
            jac = inter / uni if uni else np.nan
            overlap_records.append(
                {
                    "team_id": team_id,
                    "year": y,
                    "squad_jaccard_vs_prev_wc": jac,
                    "squad_overlap_count_vs_prev_wc": inter,
                }
            )
        prev_players = cur
        prev_year = int(y)

overlap_df = pd.DataFrame(overlap_records)
squad_feats = squad_feats.merge(
    tourn_year[["tournament_id", "year"]], on="tournament_id", how="left"
)
squad_feats = squad_feats.merge(overlap_df, on=["team_id", "year"], how="left")

squad_home = squad_feats.rename(
    columns={
        c: f"home_squad_{c}"
        for c in squad_feats.columns
        if c not in ("tournament_id", "team_id")
    }
)
squad_home = squad_home.rename(columns={"team_id": "home_team_id"})

squad_away = squad_feats.rename(
    columns={
        c: f"away_squad_{c}"
        for c in squad_feats.columns
        if c not in ("tournament_id", "team_id")
    }
)
squad_away = squad_away.rename(columns={"team_id": "away_team_id"})

base = base.merge(squad_home, on=["tournament_id", "home_team_id"], how="left")
base = base.merge(squad_away, on=["tournament_id", "away_team_id"], how="left")

base["feat_squad_age_mean_diff"] = (
    base["home_squad_age_mean"] - base["away_squad_age_mean"]
)
base["feat_squad_prior_wc_mean_diff"] = (
    base["home_squad_prior_wc_mean"] - base["away_squad_prior_wc_mean"]
)

**Coaches**

Who is listed for each team in `manager_appointments`, whether they are from the same country as the team, how many previous World Cups they have been in, a smoothed win rate from their past World Cup matches (`manager_appearances` + results), and how many times in a row they have taken this national team to the World Cup (allowing typical 4-year gaps).

In [8]:
ma = manager_appointments.merge(
    tourn_year[["tournament_id", "year"]], on="tournament_id"
)
ma = ma.merge(managers, on="manager_id", how="left")
ma_pick = ma.sort_values("manager_id").drop_duplicates(
    subset=["tournament_id", "team_id"], keep="first"
)

ma_home = ma_pick.rename(
    columns={
        "manager_id": "home_manager_id",
        "country_name": "home_manager_country_name",
        "female": "home_manager_female",
    }
)
base = base.merge(
    ma_home[
        [
            "tournament_id",
            "team_id",
            "home_manager_id",
            "home_manager_country_name",
            "home_manager_female",
        ]
    ].rename(columns={"team_id": "home_team_id"}),
    on=["tournament_id", "home_team_id"],
    how="left",
)

ma_away = ma_pick.rename(
    columns={
        "manager_id": "away_manager_id",
        "country_name": "away_manager_country_name",
        "female": "away_manager_female",
    }
)
base = base.merge(
    ma_away[
        [
            "tournament_id",
            "team_id",
            "away_manager_id",
            "away_manager_country_name",
            "away_manager_female",
        ]
    ].rename(columns={"team_id": "away_team_id"}),
    on=["tournament_id", "away_team_id"],
    how="left",
)

base["feat_home_manager_local"] = (
    _norm_host(base["home_manager_country_name"]) == _norm_host(base["home_team_name"])
).astype(int)
base["feat_away_manager_local"] = (
    _norm_host(base["away_manager_country_name"]) == _norm_host(base["away_team_name"])
).astype(int)

mgr_tourneys = (
    ma_pick[["manager_id", "tournament_id", "year"]]
    .drop_duplicates()
    .sort_values(["manager_id", "year"])
)
mgr_prior_rows = []
for mid, g in mgr_tourneys.groupby("manager_id"):
    for i, (_, r) in enumerate(g.iterrows()):
        mgr_prior_rows.append(
            {
                "manager_id": mid,
                "tournament_id": r["tournament_id"],
                "mgr_n_prior_wc": i,
            }
        )
mgr_prior_df = pd.DataFrame(mgr_prior_rows)
base = base.merge(
    mgr_prior_df.rename(columns={"manager_id": "home_manager_id"})[
        ["home_manager_id", "tournament_id", "mgr_n_prior_wc"]
    ].rename(columns={"mgr_n_prior_wc": "home_mgr_n_prior_wc"}),
    on=["home_manager_id", "tournament_id"],
    how="left",
)
base = base.merge(
    mgr_prior_df.rename(columns={"manager_id": "away_manager_id"})[
        ["away_manager_id", "tournament_id", "mgr_n_prior_wc"]
    ].rename(columns={"mgr_n_prior_wc": "away_mgr_n_prior_wc"}),
    on=["away_manager_id", "tournament_id"],
    how="left",
)

mapp = manager_appearances.merge(
    tourn_year[["tournament_id", "year"]], on="tournament_id"
).merge(
    team_appearances[["tournament_id", "match_id", "team_id", "win"]],
    on=["tournament_id", "match_id", "team_id"],
    how="left",
)


def mgr_hist_win_rate(manager_id: str, before_year: int) -> float:
    sub = mapp[(mapp["manager_id"] == manager_id) & (mapp["year"] < before_year)]
    if sub.empty or sub["win"].isna().all():
        return np.nan
    sub = sub.dropna(subset=["win"])
    return shrink_win_rate(int(sub["win"].sum()), len(sub))


base["feat_home_mgr_hist_win_rate_shrunk"] = [
    mgr_hist_win_rate(m, int(y)) if pd.notna(m) else np.nan
    for m, y in zip(base["home_manager_id"], base["year"])
]
base["feat_away_mgr_hist_win_rate_shrunk"] = [
    mgr_hist_win_rate(m, int(y)) if pd.notna(m) else np.nan
    for m, y in zip(base["away_manager_id"], base["year"])
]

streak_rows = []
for (mid, tid), g in ma_pick.sort_values("year").groupby(["manager_id", "team_id"]):
    g = g.drop_duplicates(subset=["tournament_id", "year"]).sort_values("year")
    yrs = g["year"].astype(int).tolist()
    tids = g["tournament_id"].tolist()
    run = 0
    for i, y in enumerate(yrs):
        if i == 0:
            run = 1
        else:
            gap = y - yrs[i - 1]
            run = run + 1 if gap in (1, 4) else 1
        streak_rows.append(
            {
                "manager_id": mid,
                "team_id": tid,
                "tournament_id": tids[i],
                "mgr_consecutive_wc_with_team": run,
            }
        )

streak_df = pd.DataFrame(streak_rows)
base = base.merge(
    streak_df.rename(
        columns={"manager_id": "home_manager_id", "team_id": "home_team_id"}
    )[
        [
            "home_manager_id",
            "home_team_id",
            "tournament_id",
            "mgr_consecutive_wc_with_team",
        ]
    ].rename(
        columns={"mgr_consecutive_wc_with_team": "home_mgr_consecutive_wc_with_team"}
    ),
    on=["home_manager_id", "home_team_id", "tournament_id"],
    how="left",
)
base = base.merge(
    streak_df.rename(
        columns={"manager_id": "away_manager_id", "team_id": "away_team_id"}
    )[
        [
            "away_manager_id",
            "away_team_id",
            "tournament_id",
            "mgr_consecutive_wc_with_team",
        ]
    ].rename(
        columns={"mgr_consecutive_wc_with_team": "away_mgr_consecutive_wc_with_team"}
    ),
    on=["away_manager_id", "away_team_id", "tournament_id"],
    how="left",
)

**Schedule and venue context**

- Rest days: for each team, days between this match date and their previous match date in the same tournament. 

- Referees: how many referee rows are linked to the match in `referee_appearances`.

In [9]:
sched = matches[
    [
        "match_id",
        "tournament_id",
        "home_team_id",
        "away_team_id",
        "match_dt",
        "year",
    ]
].copy()

long_sched = pd.concat(
    [
        sched.rename(
            columns={"home_team_id": "team_id", "away_team_id": "opponent_id"}
        ),
        sched.rename(
            columns={"away_team_id": "team_id", "home_team_id": "opponent_id"}
        ),
    ],
    ignore_index=True,
)
long_sched = long_sched.sort_values(
    ["tournament_id", "team_id", "match_dt", "match_id"]
)
long_sched["prev_match_dt"] = long_sched.groupby(["tournament_id", "team_id"])[
    "match_dt"
].shift(1)
long_sched["rest_days"] = (long_sched["match_dt"] - long_sched["prev_match_dt"]).dt.days

home_rest = long_sched.rename(columns={"rest_days": "home_rest_days_since_prev_match"})[
    ["match_id", "team_id", "home_rest_days_since_prev_match"]
].rename(columns={"team_id": "home_team_id"})
away_rest = long_sched.rename(columns={"rest_days": "away_rest_days_since_prev_match"})[
    ["match_id", "team_id", "away_rest_days_since_prev_match"]
].rename(columns={"team_id": "away_team_id"})

base = base.merge(home_rest, on=["match_id", "home_team_id"], how="left")
base = base.merge(away_rest, on=["match_id", "away_team_id"], how="left")
base["feat_rest_days_diff"] = (
    base["home_rest_days_since_prev_match"] - base["away_rest_days_since_prev_match"]
)

nref = referee_appearances.groupby("match_id").size().reset_index(name="n_referees")
base = base.merge(nref, on="match_id", how="left")
base["n_referees"] = base["n_referees"].fillna(0).astype(int)

**What we save and where**

- Identifiers and target

    - `match_id`, `tournament_id`, `year`, `home_team_id`, `away_team_id`, `y_win`

- Input columns
    - Built comparisons: names starting with `feat_` (e.g. home vs away historical win rate difference).

    - History blocks: `home_hist_*`, `away_hist_*` (past World Cups only).

    - Squads: `home_squad_*`, `away_squad_*` (e.g. `home_squad_n_players`, `home_squad_age_mean`, `home_squad_jaccard_vs_prev_wc`).

    - Coaches: `home_mgr_*`, `away_mgr_*`, and related `feat_*mgr*` columns.
    
    - Match setting: `stadium_capacity`, `group_stage`, `knockout_stage`, `replayed`, `replay`, rest-day columns, `n_referees`, plus `stage_name` and `group_name`.

- Writes the stable base feature products used downstream: `data/processed/class_a_match_level.parquet` and `data/processed/class_a_match_level.csv`.

In [10]:
ID_COLS = [
    "match_id",
    "tournament_id",
    "year",
    "home_team_id",
    "away_team_id",
]
LABEL = "y_win"

numeric_hist = [
    c for c in base.columns if c.startswith("home_hist_") or c.startswith("away_hist_")
]
numeric_squad = [
    c
    for c in base.columns
    if c.startswith("home_squad_") or c.startswith("away_squad_")
]
feat_prefixed = [c for c in base.columns if c.startswith("feat_")]
mgr_numeric = [
    "home_mgr_n_prior_wc",
    "away_mgr_n_prior_wc",
    "feat_home_mgr_hist_win_rate_shrunk",
    "feat_away_mgr_hist_win_rate_shrunk",
    "home_mgr_consecutive_wc_with_team",
    "away_mgr_consecutive_wc_with_team",
    "feat_home_manager_local",
    "feat_away_manager_local",
]
schedule_ctx = [
    "group_stage",
    "knockout_stage",
    "replayed",
    "replay",
    "stadium_capacity",
    "home_rest_days_since_prev_match",
    "away_rest_days_since_prev_match",
    "n_referees",
]

FEATURE_COLS = sorted(
    set(numeric_hist + numeric_squad + feat_prefixed + mgr_numeric + schedule_ctx)
)
EXPORT_COLS = ID_COLS + [LABEL] + FEATURE_COLS + ["stage_name", "group_name"]

out = base[[c for c in EXPORT_COLS if c in base.columns]].copy()

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
parquet_path = PROCESSED_DIR / "class_a_match_level.parquet"
csv_path = PROCESSED_DIR / "class_a_match_level.csv"

try:
    out.to_parquet(parquet_path, index=False)
    print("Wrote", parquet_path.relative_to(REPO_ROOT))
except Exception as e:
    print("Parquet export skipped:", e)

out.to_csv(csv_path, index=False)
print("Wrote", csv_path.relative_to(REPO_ROOT))
out.shape

Wrote data/processed/class_a_match_level.parquet
Wrote data/processed/class_a_match_level.csv


(1248, 104)

#### **Exported columns - reference by category**

What each group of variables means in `out` / `class_a_match_level.*`. Home = the home side in `matches` for that row. History features never use matches from the current tournament—only strictly earlier editions.

**Category 0 - IDs and label (6 columns)**

| Column | Meaning |
|--------|---------|
| `match_id` | Unique match |
| `tournament_id` | Unique edition (men’s and women’s; `year` matches this) |
| `year` | Calendar year of that World Cup |
| `home_team_id` / `away_team_id` | Home / away team IDs |
| `y_win` | Label: 1 if home win, 0 if draw or away win |

**Category 1 - Schedule, venue, officials (11 numeric + text)**

| Column | Meaning |
|--------|---------|
| `stage_name`, `group_name` | Stage and group labels (often one-hot encoded in modeling) |
| `group_stage`, `knockout_stage` | Group vs knockout round flags (0/1) |
| `replayed`, `replay` | Replay flags |
| `stadium_capacity` | Stadium capacity |
| `home_rest_days_since_prev_match` / `away_...` | Days since this team’s previous kickoff in the same tournament (first match may be missing) |
| `feat_rest_days_diff` | Home minus away rest days |
| `n_referees` | Number of referee rows linked to the match |

**Category 2 - Historical record: `home_hist_*` / `away_hist_*` (17 each, 34 total)**

All aggregated from `team_appearances` with `year` strictly before the current edition.

| Suffix / pattern | Meaning |
|------------------|---------|
| `n_tournaments` | Prior World Cups with at least one appearance |
| `n_matches` | Total prior World Cup matches |
| `wins`, `draws`, `losses` | Prior win / draw / loss counts |
| `goal_diff_sum` | Sum of goal difference |
| `goal_diff_per_match` | Average goal difference per match |
| `win_rate_shrunk` | Shrunk prior win rate (pulled toward 50% for small samples) |
| `ko_matches`, `ko_wins`, `ko_win_rate_shrunk` | Knockout games, wins, shrunk win rate |
| `pso_matches`, `pso_wins`, `pso_win_rate_shrunk` | Penalty shootout games (often sparse / missing) |
| `et_matches`, `et_rate` | Extra-time games and rate |
| `tournaments_with_ko_game`, `frac_tournaments_reached_ko` | Editions with at least one KO match and share of editions |

**Category 3 - Comparisons and identity: `feat_*` (12 columns)**

| Column | Meaning |
|--------|---------|
| `feat_hist_win_rate_diff` | Home minus away shrunk historical win rate |
| `feat_hist_goal_diff_per_match_diff` | Home minus away prior avg goal difference |
| `feat_home_hist_win_rate_vs_away_conf` | Home team’s shrunk win rate vs opponents from the current away team’s confederation (prior cups only) |
| `feat_away_hist_win_rate_vs_home_conf` | Same for away vs home’s confederation |
| `feat_same_confederation` | Same confederation (0/1) |
| `feat_home_is_host_nation` / `feat_away_is_host_nation` | Host nation team flags |
| `feat_squad_age_mean_diff` / `feat_squad_prior_wc_mean_diff` | Squad mean age diff; mean prior-WC-count diff |
| `feat_home_manager_local` / `feat_away_manager_local` | Manager country matches team name (rough rule) |
| `feat_home_mgr_hist_win_rate_shrunk` / `feat_away_...` | Manager’s shrunk win rate in prior World Cup matches |
| `feat_rest_days_diff` | See category 1 |

**Category 4 — Squads: `home_squad_*` / `away_squad_*` (18 per side)**

From current `squads` + `players`. Player prior tournament counts only include editions before this row’s `year`.

| Suffix | Meaning |
|--------|---------|
| `n_players` | Squad size |
| `age_mean`, `age_median`, `age_std`, `age_min`, `age_max` | Age vs tournament start |
| `n_gk`, `n_df`, `n_mf`, `n_fw` | Counts by position flags |
| `prior_wc_mean`, `prior_wc_median` | Mean / median prior World Cups played per player |
| `share_any_prior_wc` | Share with at least one prior World Cup |
| `share_ge2_prior_wc` | Share with ≥2 prior World Cups |
| `squad_jaccard_vs_prev_wc`, `squad_overlap_count_vs_prev_wc` | Jaccard and intersection vs this team’s previous World Cup squad (column names may repeat `squad_`) |
| `year` (e.g. `home_squad_year`) | Copy of edition year for checks |

**Category 5 — Manager counts (4 columns, no `feat_` prefix)**

| Column | Meaning |
|--------|---------|
| `home_mgr_n_prior_wc` / `away_...` | Index count of prior World Cups this manager had before this edition (0, 1, …) |
| `home_mgr_consecutive_wc_with_team` / `away_...` | Streak length with this national team (allows typical 4-year gaps) |

**Summary:** 0 IDs/label · 1 schedule/venue/officials · 2 history (×2 sides) · 3 `feat_` comparisons · 4 squads (×2) · 5 manager counts; plus text `stage_name` / `group_name`. Exact column count = `out.shape[1]` after running the export cell (may vary slightly if the DB build differs).

In [11]:
# Column counts by prefix (run after the export cell that creates `out`)
from IPython.display import display

inv_rows = [
    (
        "IDs and label",
        [
            c
            for c in out.columns
            if c
            in (
                "match_id",
                "tournament_id",
                "year",
                "home_team_id",
                "away_team_id",
                "y_win",
            )
        ],
    ),
    (
        "Schedule / venue / officials",
        [
            c
            for c in out.columns
            if c
            in (
                "group_stage",
                "knockout_stage",
                "replayed",
                "replay",
                "stadium_capacity",
                "home_rest_days_since_prev_match",
                "away_rest_days_since_prev_match",
                "n_referees",
            )
            or c == "feat_rest_days_diff"
        ],
    ),
    ("Text schedule", [c for c in out.columns if c in ("stage_name", "group_name")]),
    ("History home", [c for c in out.columns if c.startswith("home_hist_")]),
    ("History away", [c for c in out.columns if c.startswith("away_hist_")]),
    ("Comparisons feat_", [c for c in out.columns if c.startswith("feat_")]),
    ("Squad home", [c for c in out.columns if c.startswith("home_squad_")]),
    ("Squad away", [c for c in out.columns if c.startswith("away_squad_")]),
    (
        "Manager counts",
        [
            c
            for c in out.columns
            if c
            in (
                "home_mgr_n_prior_wc",
                "away_mgr_n_prior_wc",
                "home_mgr_consecutive_wc_with_team",
                "away_mgr_consecutive_wc_with_team",
            )
        ],
    ),
]
inv_df = pd.DataFrame(
    [{"category": name, "n_columns": len(cols)} for name, cols in inv_rows]
)
display(inv_df)
print("Total columns out.shape:", out.shape)

,category,n_columns
0,IDs and label,6
1,Schedule / venue / officials,9
2,Text schedule,2
3,History home,18
4,History away,18
5,Comparisons feat_,14
6,Squad home,17
7,Squad away,17
8,Manager counts,4


Total columns out.shape: (1248, 104)


In [12]:
con.close()